# LSTM Experiment C Historical Weather

**24-hour input lookback → next-hour EV charging demand.**

Extracted from `Untitled0.ipynb`. Original execution outputs are retained. Run cells in order in Google Colab; confirm Google Drive paths before executing.

## Data loading, preprocessing, model training, evaluation and saved results

## Notebook Setup

In [ ]:

# ============================================================
# EXPERIMENT I
# LSTM - HISTORICAL CHARGING DEMAND + WEATHER
# 24-HOUR LOOKBACK -> NEXT 1-HOUR FORECAST
# ============================================================

# ============================================================


## 1. Import Libraries and Mount Google Drive

In [ ]:
# 1. IMPORT LIBRARIES AND MOUNT GOOGLE DRIVE
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

import os
import json
import random
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

from tensorflow.keras import Sequential
from tensorflow.keras.layers import (
    Input,
    LSTM,
    Dropout,
    Dense
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
    ReduceLROnPlateau
)

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

print("TensorFlow version:", tf.__version__)


# ============================================================


## 2. Experiment Configuration

In [ ]:
# 2. EXPERIMENT CONFIGURATION
# ============================================================

ROOT = "/content/drive/MyDrive/EV_Load_Forecasting_FYP"

DATASET_PATH = os.path.join(
    ROOT,
    "Processed Data",
    "Hourly",
    "experiment_c_final_2019.csv"
)

EXP_DIR = os.path.join(
    ROOT,
    "Experiments",
    "LSTM_Experiment_I_Historical_Weather_2019"
)

os.makedirs(EXP_DIR, exist_ok=True)

LOOKBACK = 24
BATCH_SIZE = 32
EPOCHS = 200
LEARNING_RATE = 0.001

MODEL_PATH = os.path.join(
    EXP_DIR,
    "lstm_experiment_i_best.keras"
)

WEATHER_COLS = [
    "clouds_lag1d",
    "dewpt_lag1d",
    "ghi_lag1d",
    "max_uv_lag1d",
    "precip_lag1d",
    "rh_lag1d",
    "slp_lag1d",
    "temp_lag1d",
    "wind_gust_spd_lag1d",
    "wind_spd_lag1d"
]

print("\nExperiment directory:", EXP_DIR)


# ============================================================


## 3. Load and Inspect Dataset

In [ ]:
# 3. LOAD DATASET
# ============================================================

df = pd.read_csv(
    DATASET_PATH,
    parse_dates=["date_hour"]
)

df = (
    df.sort_values("date_hour")
    .reset_index(drop=True)
)

required_columns = [
    "date_hour",
    "hourly_energy_kwh"
] + WEATHER_COLS

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

assert not missing_columns, (
    f"Missing columns: {missing_columns}"
)

df = df[required_columns].copy()

numeric_cols = [
    "hourly_energy_kwh"
] + WEATHER_COLS

for col in numeric_cols:
    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )

print("\n" + "=" * 70)
print("EXPERIMENT I - DATASET INFORMATION")
print("=" * 70)

print("Dataset shape:", df.shape)

print("\nDate range:")
print("Start:", df["date_hour"].min())
print("End:", df["date_hour"].max())

print("\nMissing values:")
print(df.isna().sum())

print("\nFirst 5 rows:")
print(df.head())

assert len(df) == 8016
assert not df["date_hour"].duplicated().any()

expected_dates = pd.date_range(
    df["date_hour"].min(),
    df["date_hour"].max(),
    freq="h"
)

assert np.array_equal(
    expected_dates.to_numpy(),
    df["date_hour"].to_numpy()
)


# ============================================================


## 4. Chronological Data Splitting

In [ ]:
# 4. CHRONOLOGICAL SPLITTING
# ============================================================

n_total = len(df)

train_end = int(n_total * 0.70)
val_end = int(n_total * 0.85)

print("\n" + "=" * 70)
print("EXPERIMENT I - CHRONOLOGICAL SPLITTING")
print("=" * 70)

for name, split in [
    ("Training", df.iloc[:train_end]),
    ("Validation", df.iloc[train_end:val_end]),
    ("Testing", df.iloc[val_end:])
]:
    print(f"\n{name}")
    print("Rows:", len(split))
    print("Start:", split["date_hour"].min())
    print("End:", split["date_hour"].max())

    print(
        "Missing demand:",
        split["hourly_energy_kwh"].isna().sum()
    )

assert train_end == 5611
assert val_end - train_end == 1202
assert n_total - val_end == 1203


# ============================================================


## 5. Missing Value Handling

In [ ]:
# 5. MISSING VALUE HANDLING
# ============================================================

df_processed = df.copy()

# Preserve original observed charging demand
df_processed["target_original"] = (
    df_processed["hourly_energy_kwh"]
)

# Causal forward filling for historical demand
df_processed["demand_input"] = (
    df_processed["hourly_energy_kwh"].ffill()
)

# Weather features are already lagged by one day.
# Forward-fill missing weather inputs causally.
df_processed[WEATHER_COLS] = (
    df_processed[WEATHER_COLS].ffill()
)

# Handle any leading weather NaNs using training-only medians
# (without consulting validation or testing distributions).
for col in WEATHER_COLS:

    if df_processed[col].isna().any():

        train_median = (
            df_processed.iloc[:train_end][col].median()
        )

        if pd.isna(train_median):
            raise ValueError(
                f"No observed training values for {col}"
            )

        df_processed[col] = (
            df_processed[col].fillna(train_median)
        )

print("\n" + "=" * 70)
print("EXPERIMENT I - MISSING VALUE HANDLING")
print("=" * 70)

print(
    df_processed[
        ["target_original", "demand_input"] + WEATHER_COLS
    ].isna().sum()
)

assert df_processed["demand_input"].notna().all()
assert df_processed[WEATHER_COLS].notna().all().all()

df_processed.to_csv(
    os.path.join(
        EXP_DIR,
        "processed_dataset.csv"
    ),
    index=False
)


# ============================================================


## 6. Feature Scaling (Training Data Only)

In [ ]:
# 6. FEATURE SCALING - TRAINING DATA ONLY
# ============================================================

feature_cols = [
    "demand_input"
] + WEATHER_COLS

n_features = len(feature_cols)

scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

scaler_X.fit(
    df_processed.iloc[:train_end][feature_cols]
)

train_targets = (
    df_processed.iloc[:train_end][["target_original"]]
    .dropna()
)

scaler_y.fit(train_targets)

X_all_scaled = scaler_X.transform(
    df_processed[feature_cols]
).astype(np.float32)

y_all_scaled = np.full(
    len(df_processed),
    np.nan,
    dtype=np.float32
)

valid_mask = (
    df_processed["target_original"]
    .notna()
    .to_numpy()
)

y_all_scaled[valid_mask] = (
    scaler_y.transform(
        df_processed.loc[
            valid_mask,
            ["target_original"]
        ]
    )
    .ravel()
    .astype(np.float32)
)

joblib.dump(
    scaler_X,
    os.path.join(EXP_DIR, "scaler_X.joblib")
)

joblib.dump(
    scaler_y,
    os.path.join(EXP_DIR, "scaler_y.joblib")
)

print("\n" + "=" * 70)
print("EXPERIMENT I - SCALING")
print("=" * 70)

print("Input features:", feature_cols)
print("Number of features:", n_features)
print("Scaled X shape:", X_all_scaled.shape)
print("Scaled y shape:", y_all_scaled.shape)

assert X_all_scaled.shape == (8016, 11)
assert y_all_scaled.shape == (8016,)
assert np.isfinite(X_all_scaled).all()
assert np.isnan(y_all_scaled).sum() == 206


# ============================================================


## 7. Generate LSTM Input Sequences

In [ ]:
# 7. GENERATE LSTM SEQUENCES
# ============================================================

def generate_sequences(
    X_data,
    y_data,
    timestamps,
    lookback,
    train_end,
    val_end
):

    containers = {
        "train": {"X": [], "y": [], "time": []},
        "val": {"X": [], "y": [], "time": []},
        "test": {"X": [], "y": [], "time": []}
    }

    skipped_missing = 0

    for target_idx in range(lookback, len(y_data)):

        y_target = y_data[target_idx]

        if np.isnan(y_target):
            skipped_missing += 1
            continue

        X_window = X_data[
            target_idx-lookback:target_idx
        ]

        if not np.isfinite(X_window).all():
            continue

        if target_idx < train_end:
            split = "train"
        elif target_idx < val_end:
            split = "val"
        else:
            split = "test"

        containers[split]["X"].append(X_window)
        containers[split]["y"].append(y_target)
        containers[split]["time"].append(
            timestamps[target_idx]
        )

    result = {}

    for split in ["train", "val", "test"]:

        result[split] = {
            "X": np.asarray(
                containers[split]["X"],
                dtype=np.float32
            ).reshape(-1, lookback, X_data.shape[1]),

            "y": np.asarray(
                containers[split]["y"],
                dtype=np.float32
            ).reshape(-1, 1),

            "time": pd.DatetimeIndex(
                containers[split]["time"]
            )
        }

    print("\nSkipped missing targets:", skipped_missing)

    return result


data = generate_sequences(
    X_all_scaled,
    y_all_scaled,
    df_processed["date_hour"].to_numpy(),
    LOOKBACK,
    train_end,
    val_end
)

X_train = data["train"]["X"]
y_train = data["train"]["y"]
time_train = data["train"]["time"]

X_val = data["val"]["X"]
y_val = data["val"]["y"]
time_val = data["val"]["time"]

X_test = data["test"]["X"]
y_test = data["test"]["y"]
time_test = data["test"]["time"]

print("\n" + "=" * 70)
print("EXPERIMENT I - LSTM SEQUENCE SHAPES")
print("=" * 70)

for name, X, y, times in [
    ("Training", X_train, y_train, time_train),
    ("Validation", X_val, y_val, time_val),
    ("Testing", X_test, y_test, time_test)
]:
    print(f"\n{name}")
    print("X shape:", X.shape)
    print("y shape:", y.shape)
    print("First target:", times.min())
    print("Last target:", times.max())

assert X_train.shape == (5403, 24, 11)
assert X_val.shape == (1187, 24, 11)
assert X_test.shape == (1196, 24, 11)


# ============================================================


## 8. Build LSTM Model

In [ ]:
# 8. BUILD LSTM MODEL
# SAME ARCHITECTURE AS EXPERIMENT F
# ============================================================

model = Sequential([
    Input(shape=(LOOKBACK, n_features)),

    LSTM(64, return_sequences=True),
    Dropout(0.2),

    LSTM(32, return_sequences=True),
    Dropout(0.2),

    LSTM(16),
    Dropout(0.2),

    Dense(1)
])

model.compile(
    optimizer=Adam(
        learning_rate=LEARNING_RATE,
        clipnorm=1.0
    ),
    loss="mse",
    metrics=["mae"]
)

print("\n" + "=" * 70)
print("EXPERIMENT I - LSTM ARCHITECTURE")
print("=" * 70)

model.summary()

print("\nTotal parameters:", model.count_params())


# ============================================================


## 9. Train LSTM Model

In [ ]:
# 9. TRAIN LSTM MODEL
# ============================================================

callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=20,
        restore_best_weights=True,
        mode="min",
        verbose=1
    ),

    ModelCheckpoint(
        filepath=MODEL_PATH,
        monitor="val_loss",
        save_best_only=True,
        mode="min",
        verbose=1
    ),

    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=10,
        min_lr=1e-6,
        mode="min",
        verbose=1
    )
]

history = model.fit(
    X_train,
    y_train,

    validation_data=(
        X_val,
        y_val
    ),

    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    shuffle=False,

    callbacks=callbacks,
    verbose=1
)

history_df = pd.DataFrame(
    history.history
)

history_df.to_csv(
    os.path.join(
        EXP_DIR,
        "training_history.csv"
    ),
    index=False
)

best_epoch = (
    np.argmin(history.history["val_loss"]) + 1
)

print("\n" + "=" * 70)
print("EXPERIMENT I - TRAINING SUMMARY")
print("=" * 70)

print("Total epochs:",
      len(history.history["loss"]))

print("Best epoch:", best_epoch)

print(
    "Best validation MSE (scaled):",
    round(
        float(np.min(history.history["val_loss"])),
        6
    )
)


# ============================================================


## 10. Load Best Saved Model

In [ ]:
# 10. LOAD BEST MODEL
# ============================================================

best_model = tf.keras.models.load_model(
    MODEL_PATH
)


# ============================================================


## 11. Define Evaluation Function

In [ ]:
# 11. EVALUATION FUNCTION
# ============================================================

def evaluate_model(
    model,
    X,
    y,
    timestamps,
    split_name
):

    y_pred_scaled = model.predict(
        X,
        verbose=0
    )

    y_pred = scaler_y.inverse_transform(
        y_pred_scaled.reshape(-1, 1)
    ).ravel()

    y_actual = scaler_y.inverse_transform(
        y.reshape(-1, 1)
    ).ravel()

    rmse = np.sqrt(
        mean_squared_error(y_actual, y_pred)
    )

    mae = mean_absolute_error(
        y_actual, y_pred
    )

    r2 = r2_score(
        y_actual, y_pred
    )

    prediction_df = pd.DataFrame({
        "date_hour": timestamps,
        "actual_kwh": y_actual,
        "predicted_kwh": y_pred
    })

    prediction_df["error_kwh"] = (
        prediction_df["predicted_kwh"]
        - prediction_df["actual_kwh"]
    )

    metrics = {
        "Dataset": split_name,
        "Samples": len(y_actual),
        "RMSE (kWh)": rmse,
        "MAE (kWh)": mae,
        "R² Score": r2
    }

    return metrics, prediction_df


# ============================================================


## 12. Evaluate Training, Validation and Testing

In [ ]:
# 12. EVALUATE TRAINING / VALIDATION / TESTING
# ============================================================

metrics_train, predictions_train = evaluate_model(
    best_model,
    X_train,
    y_train,
    time_train,
    "Training"
)

metrics_val, predictions_val = evaluate_model(
    best_model,
    X_val,
    y_val,
    time_val,
    "Validation"
)

metrics_test, predictions_test = evaluate_model(
    best_model,
    X_test,
    y_test,
    time_test,
    "Testing"
)

metrics_df = pd.DataFrame([
    metrics_train,
    metrics_val,
    metrics_test
])

print("\n" + "=" * 70)
print("EXPERIMENT I - PERFORMANCE SUMMARY")
print("=" * 70)

print(
    metrics_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)


# ============================================================


## 13. Save Evaluation Results

In [ ]:
# 13. SAVE EVALUATION RESULTS
# ============================================================

metrics_df.to_csv(
    os.path.join(
        EXP_DIR,
        "all_split_metrics.csv"
    ),
    index=False
)

for name, predictions in [
    ("training", predictions_train),
    ("validation", predictions_val),
    ("test", predictions_test)
]:

    predictions.to_csv(
        os.path.join(
            EXP_DIR,
            f"{name}_predictions.csv"
        ),
        index=False
    )


# ============================================================


## 14. Plot Training and Validation Loss

In [ ]:
# 14. PLOT TRAINING AND VALIDATION LOSS
# ============================================================

plt.figure(figsize=(10, 5))

plt.plot(
    history.history["loss"],
    label="Training Loss"
)

plt.plot(
    history.history["val_loss"],
    label="Validation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("MSE (Scaled)")
plt.title(
    "Experiment I - Training vs Validation Loss"
)

plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig(
    os.path.join(
        EXP_DIR,
        "training_loss.png"
    ),
    dpi=300
)

plt.show()


# ============================================================


## 15. Plot Actual vs Predicted Test Demand

In [ ]:
# 15. PLOT ACTUAL VS PREDICTED TEST DEMAND
# ============================================================

plt.figure(figsize=(14, 5))

plt.plot(
    predictions_test["date_hour"],
    predictions_test["actual_kwh"],
    label="Actual",
    linewidth=1
)

plt.plot(
    predictions_test["date_hour"],
    predictions_test["predicted_kwh"],
    label="LSTM I - Historical + Weather",
    linewidth=1
)

plt.xlabel("Date")
plt.ylabel("Charging Demand (kWh)")
plt.title(
    "Experiment I - Actual vs Predicted Test Demand"
)

plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig(
    os.path.join(
        EXP_DIR,
        "test_actual_vs_predicted.png"
    ),
    dpi=300
)

plt.show()


# ============================================================


## 16. Plot First 168 Test Observations

In [ ]:
# 16. PLOT FIRST 168 TEST OBSERVATIONS
# ============================================================

week_df = predictions_test.head(168)

plt.figure(figsize=(14, 5))

plt.plot(
    week_df["date_hour"],
    week_df["actual_kwh"],
    label="Actual",
    linewidth=1.5
)

plt.plot(
    week_df["date_hour"],
    week_df["predicted_kwh"],
    label="LSTM I",
    linewidth=1.5
)

plt.xlabel("Date")
plt.ylabel("Charging Demand (kWh)")
plt.title(
    "Experiment I - First 168 Test Observations"
)

plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig(
    os.path.join(
        EXP_DIR,
        "test_first_168_hours.png"
    ),
    dpi=300
)

plt.show()


# ============================================================


## 17. Matched-Timestamp Model Comparison

In [ ]:
# 17. MATCHED-TIMESTAMP COMPARISON
# EXPERIMENT C VS F VS H VS I
# ============================================================

comparison_sources = {
    "LSTM C - All Features": os.path.join(
        ROOT,
        "Experiments",
        "LSTM_Experiment_C_2019",
        "test_predictions.csv"
    ),

    "LSTM F - Historical Only": os.path.join(
        ROOT,
        "Experiments",
        "LSTM_Experiment_F_Historical_Only_2019",
        "test_predictions.csv"
    ),

    "LSTM H - Historical + Traffic": os.path.join(
        ROOT,
        "Experiments",
        "LSTM_Experiment_H_Historical_Traffic_2019",
        "test_predictions.csv"
    ),

    "LSTM I - Historical + Weather": os.path.join(
        EXP_DIR,
        "test_predictions.csv"
    )
}

comparison_frames = {}

for model_name, path in comparison_sources.items():

    if not os.path.exists(path):
        print(f"Skipping missing file: {path}")
        continue

    temp = pd.read_csv(
        path,
        parse_dates=["date_hour"]
    )

    temp.columns = [
        col.strip().lower()
        for col in temp.columns
    ]

    required = [
        "date_hour",
        "actual_kwh",
        "predicted_kwh"
    ]

    assert all(
        col in temp.columns
        for col in required
    ), f"Missing columns in {path}"

    temp = temp[required].copy()

    assert not temp["date_hour"].duplicated().any()

    comparison_frames[model_name] = temp


# Find common timestamps
common_times = None

for temp in comparison_frames.values():

    times = set(temp["date_hour"])

    if common_times is None:
        common_times = times
    else:
        common_times &= times

common_times = sorted(common_times)

print("\n" + "=" * 70)
print("MATCHED-TIMESTAMP COMPARISON")
print("=" * 70)

print("Included models:", list(comparison_frames.keys()))
print("Common testing timestamps:", len(common_times))

assert len(comparison_frames) >= 2
assert len(common_times) > 0

comparison_results = []
reference_actual = None

for model_name, temp in comparison_frames.items():

    matched = (
        temp[
            temp["date_hour"].isin(common_times)
        ]
        .sort_values("date_hour")
        .reset_index(drop=True)
    )

    actual = matched["actual_kwh"].to_numpy()
    predicted = matched["predicted_kwh"].to_numpy()

    if reference_actual is None:
        reference_actual = actual
    else:
        assert np.allclose(
            reference_actual,
            actual,
            rtol=1e-5,
            atol=1e-3
        ), "Actual target values differ."

    comparison_results.append({
        "Model": model_name,
        "Samples": len(matched),
        "RMSE (kWh)": np.sqrt(
            mean_squared_error(actual, predicted)
        ),
        "MAE (kWh)": mean_absolute_error(
            actual, predicted
        ),
        "R² Score": r2_score(
            actual, predicted
        )
    })

comparison_df = pd.DataFrame(
    comparison_results
)

print(
    comparison_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

comparison_df.to_csv(
    os.path.join(
        EXP_DIR,
        "comparison_C_F_H_I_matched_metrics.csv"
    ),
    index=False
)


# ============================================================


## 18. Save Experiment Configuration

In [ ]:
# 18. WEATHER FEATURE CONTRIBUTION
# ============================================================

if (
    "LSTM F - Historical Only"
    in comparison_df["Model"].values
):

    rmse_f = comparison_df.loc[
        comparison_df["Model"] ==
        "LSTM F - Historical Only",
        "RMSE (kWh)"
    ].iloc[0]

    rmse_i = comparison_df.loc[
        comparison_df["Model"] ==
        "LSTM I - Historical + Weather",
        "RMSE (kWh)"
    ].iloc[0]

    improvement = (
        (rmse_f - rmse_i) / rmse_f
    ) * 100

    print("\n" + "=" * 70)
    print("WEATHER FEATURE CONTRIBUTION")
    print("=" * 70)

    print(
        f"RMSE improvement from adding weather: "
        f"{improvement:.2f}%"
    )

    if improvement > 0:
        print(
            "Adding weather reduced RMSE on the matched test set."
        )
    elif improvement < 0:
        print(
            "Adding weather increased RMSE on the matched test set."
        )
    else:
        print(
            "No RMSE difference was observed."
        )


# ============================================================
# 19. SAVE EXPERIMENT CONFIGURATION
# ============================================================

config = {
    "experiment": "LSTM I",
    "dataset": "Dataset 4 - 2019",
    "features": feature_cols,
    "lookback": LOOKBACK,
    "forecast_horizon": 1,
    "split": [0.70, 0.15, 0.15],
    "architecture": "LSTM 64-32-16",
    "dropout": 0.2,
    "loss": "mse",
    "learning_rate": LEARNING_RATE,
    "batch_size": BATCH_SIZE,
    "max_epochs": EPOCHS,
    "seed": SEED,
    "test_samples": len(y_test),
    "matched_comparison_samples": len(common_times)
}

with open(
    os.path.join(
        EXP_DIR,
        "manifest.json"
    ),
    "w"
) as f:
    json.dump(config, f, indent=4)

print("\n" + "=" * 70)
print("EXPERIMENT I COMPLETED SUCCESSFULLY")
print("=" * 70)

print("Best model:", MODEL_PATH)
print("Results directory:", EXP_DIR)
